# 00 · Setup and data download

Checks the environment and downloads the Roboflow dataset into `data/` (local) or `MyDrive/cell-image-counting/data/` (Colab). Run this once per machine or Drive.

**Local setup (once):**
```bash
conda env create -f environment.yml && conda activate cellcount
python -m ipykernel install --user --name cellcount --display-name "Python (cellcount)"
nbstripout --install   # strip notebook outputs on commit
cp .env.example .env   # then fill in ROBOFLOW_EXPORT_URL
```
Then select the **Python (cellcount)** kernel.

**Colab:** open this notebook from GitHub (File > Open notebook > GitHub), add the secrets, and Run all.

In [1]:
# --- Bootstrap (identical in every notebook). Does nothing when run locally. ---
# On Colab: mounts Google Drive (data/ and outputs/ live there), clones or updates the repo,
# and installs the `cellcount` utils. Secrets: add ROBOFLOW_EXPORT_URL (and GH_TOKEN if the
# repo is private) under the key icon in the left sidebar, with notebook access enabled.
import os, subprocess, sys

REPO_URL = "https://github.com/singhp99/cell-image-counting.git"
BRANCH = "main"  # set to a feature branch to try unmerged work

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    repo = "/content/cell-image-counting"
    try:
        url = REPO_URL.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")
    except Exception:
        url = REPO_URL
    cmd = ["git", "-C", repo, "pull", "-q"] if os.path.exists(repo) else ["git", "clone", "-q", "-b", BRANCH, url, repo]
    if subprocess.run(cmd).returncode != 0:
        raise RuntimeError("git clone/pull failed: check BRANCH and the GH_TOKEN secret")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", repo], check=True)
    sys.path.insert(0, f"{repo}/src")  # import without restarting the runtime
    os.chdir(repo)

## Environment check

In [2]:
import platform
from cellcount.config import get_paths, is_colab

paths = get_paths()
print("Python   :", platform.python_version())
print("On Colab :", is_colab())
print("Data     :", paths.data)
print("Outputs  :", paths.outputs)

Python   : 3.12.14
On Colab : False
Data     : /Users/dingshandeng/project/2-project_AInML/cell-image-counting/data
Outputs  : /Users/dingshandeng/project/2-project_AInML/cell-image-counting/outputs


## Download

The default source is the **raw export link** of our Roboflow fork: the 500 original images, with no resizing or augmentation, stored as the `ROBOFLOW_EXPORT_URL` secret.

Don't use the public project's version 1 (~1,200 images). Its 350 training images are augmented 3× and every image is stretched to 640×640, so it holds only 500 unique images.

Once someone generates a version in the fork (Roboflow > Generate, with no preprocessing or augmentation), the API route below works too: set `API_VERSION` and add the `ROBOFLOW_API_KEY` secret.

In [3]:
from cellcount.config import get_secret
from cellcount.data import download_export, download_roboflow

API_VERSION = None  # e.g. 1, once a version is generated in the fork

if API_VERSION is None:
    root = download_export(get_secret("ROBOFLOW_EXPORT_URL"), paths.data)
else:
    root = download_roboflow(get_secret("ROBOFLOW_API_KEY"), paths.data, version=API_VERSION)
root

Dataset already present at /Users/dingshandeng/project/2-project_AInML/cell-image-counting/data/roboflow-cell-counting-raw-coco


PosixPath('/Users/dingshandeng/project/2-project_AInML/cell-image-counting/data/roboflow-cell-counting-raw-coco')

## Sanity check

In [4]:
from cellcount.data import load_coco

images, boxes = load_coco(root)
print(images.groupby("split").size().to_string(), "\n")
print("classes:", boxes["category"].value_counts().to_dict())
print("cells per image:", images["count"].describe().round(1).to_dict())

split
test      50
train    350
valid    100 

classes: {'Live': 25691, 'Dead': 1024}
cells per image: {'count': 500.0, 'mean': 53.4, 'std': 14.5, 'min': 23.0, '25%': 41.8, '50%': 54.0, '75%': 65.0, 'max': 91.0}


Next: `01_eda.ipynb`.